# Lab 9 — Local Ollama Chatbot (Terminal Edition)

## Learning Goals

- Understand how LLMs work locally via Ollama
- Implement Ollama in a Python application
- Manage conversation history for multi-turn chat
- Run an LLM efficiently on limited hardware

---
## Step 1 — Install Ollama

Download and install Ollama from: https://ollama.com/download

**Windows:** Use Git Bash or WSL. Do **not** use PowerShell or the default Command Prompt. If you are in VS Code, double-check that your terminal is set to Git Bash.

**Mac / Linux:** Use your regular terminal.

Once installed, pull the model by running this in your terminal:

```bash
ollama pull llama3.2:3b
```

This downloads the model (~2 GB). Once complete, test it with:

```bash
ollama run llama3.2:3b
```

Try a prompt like `What should I eat for lunch?` — if it responds, it works. Exit with `/bye` or `Ctrl + D`.

> **Why llama3.2:3b?** It's the most capable model that fits comfortably in 8 GB of RAM. It's a significant step up from tinyllama in reasoning and instruction-following.

---
## Step 2 — Set Up Your Python Environment

Run the following in your terminal to create a virtual environment and install all required packages:

```bash
python3 -m venv venv
source venv/bin/activate
pip install ollama
pip freeze > requirements.txt
```

Or use the cell below to install directly into your current environment.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "ollama"], check=True)

---
## Step 3 — Test Ollama in Python

Before building the full chatbot, verify that Ollama works from inside Python.

**Important:** Make sure the Ollama service is running before executing this cell. In a separate terminal, run:
```bash
ollama serve
```

In [ ]:
from ollama import chat

response = chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)

print(response.message.content)

---
## Step 4 — Understanding Conversation History

Unlike a single question-answer exchange, a real chatbot needs **memory** — it must remember what was said earlier in the conversation.

Ollama (like most LLM APIs) is **stateless**: each call knows nothing about previous calls. To simulate memory, we pass the full conversation history with every request:

```
[ Your Python script ]
        |
        |  chat(model, messages=[...full history...])
        ↓
  [ Ollama: llama3.2:3b ]
        |
        |  response
        ↓
[ Your Python script ] → appends reply to history → loop
```

Each message in the history has a `role` (`user` or `assistant`) and `content` (the text). By appending both sides of every exchange, the model can refer back to earlier context.

---
## Step 5 — Build the Terminal Chatbot (chat.py)

The cell below writes `chat.py` to disk. This is a self-contained terminal chatbot that:
- Keeps a running conversation history for context
- Loops until the user types `quit`
- Requires no browser, no Flask, and no internet

In [ ]:
chat_code = """from ollama import chat

print("Chatbot ready. Type 'quit' to exit.\n")

history = []

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    history.append({"role": "user", "content": user_input})

    response = chat(
        model="llama3.2:3b",
        messages=history,
    )

    reply = response.message.content
    history.append({"role": "assistant", "content": reply})

    print(f"\nBot: {reply}\n")
"""

with open("chat.py", "w") as f:
    f.write(chat_code)

print("chat.py written.")

---
## Step 6 — Run the Chatbot

Open **two terminals**:

**Terminal 1** — start the Ollama service:
```bash
ollama serve
```

**Terminal 2** — run the chatbot:
```bash
python chat.py
```

You should see:
```
Chatbot ready. Type 'quit' to exit.

You: 
```

Type any message and press Enter. The first response may take 10–20 seconds while the model loads into memory — subsequent replies will be faster.

> **RAM tip:** Close your browser and any unused apps before running. On a clean boot with nothing else open, you should have ~2.5–3 GB free, which gives llama3.2:3b comfortable headroom.

---
## Step 7 — Verify the File Exists

Run the cell below to confirm that `chat.py` was written successfully.

In [ ]:
import os

files = ["chat.py"]

for f in files:
    status = "EXISTS" if os.path.exists(f) else "MISSING"
    print(f"{f}: {status}")